In [1]:
!uv pip install torchgeo

Using Python 3.13.15 environment at: /usr
Resolved 79 packages in 1.34s
Prepared 15 packages in 3.91s
Installed 15 packages in 179ms
 + aenum==3.1.17
 + hydra-core==1.3.7
 + jsonargparse==4.52.0
 + jsonnet==0.22.0
 + kornia==0.8.3
 + kornia-rs==0.2.0
 + lightly==1.5.26
 + lightly-utils==0.0.2
 + lightning==2.6.6
 + lightning-utilities==0.15.3
 + pytorch-lightning==2.6.6
 + segmentation-models-pytorch==0.5.0
 + torchgeo==0.10.1
 + torchmetrics==1.9.0
 + typeshed-client==2.13.0


In [2]:
import numpy as np
import gc

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt

from torchgeo.trainers import SemanticSegmentationTask

In [3]:
from google.colab import drive
drive.mount('/content/drive')
base_dir = "/content/drive/MyDrive/informal_settlement_segmentation"
data_dir = f"{base_dir}/data/deep_learning"
best_model_dir = f"{base_dir}/checkpoints/11/best.ckpt"

Mounted at /content/drive


# Load the best checkpoint model (val loss = 0.483)

In [4]:
temp_task = SemanticSegmentationTask(
    model="unet",
    backbone="resnet18",
    weights=None,
    in_channels=13,
    task="binary",
    loss="jaccard",
    lr=1e-4,
    freeze_backbone=False,
)

temp_checkpoint = torch.load(best_model_dir, map_location="cpu")
temp_task.load_state_dict(temp_checkpoint["state_dict"])
best_model = temp_task.model

/tmp/ipykernel_3911/119242896.py:1: DeprecationWarning: Use torchgeo.tasks.SemanticSegmentation instead
  temp_task = SemanticSegmentationTask(


# Evaluation Function (use batch-wise evaluation otherwise out-of-memory error)

In [5]:
def evaluate_model(patches, model, threshold=0.5):
    batch_size = 32

    counts = {
        0: {"tp": 0, "fp": 0, "fn": 0},
        1: {"tp": 0, "fp": 0, "fn": 0},
    }

    metrics_report = {}

    for i in range(0, len(patches), batch_size):

        batch = patches[i:i + batch_size].float()
        batch = F.interpolate(batch, size=(224, 224), mode="nearest")

        x = batch[:, :-1]
        y_true = batch[:, -1].long()

        logits = model.predict(x).squeeze(1)
        y_pred = (torch.sigmoid(logits) >= threshold).long()

        for c in [0, 1]:

            tp = ((y_true == c) & (y_pred == c)).sum().item()
            fp = ((y_true != c) & (y_pred == c)).sum().item()
            fn = ((y_true == c) & (y_pred != c)).sum().item()

            counts[c]["tp"] += tp
            counts[c]["fp"] += fp
            counts[c]["fn"] += fn

        del batch, x, y_true, logits, y_pred

    for c in [0, 1]:

        tp = counts[c]["tp"]
        fp = counts[c]["fp"]
        fn = counts[c]["fn"]

        iou       = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        precision = tp / (tp + fp     ) if (tp + fp     ) > 0 else 0.0
        recall    = tp / (tp +      fn) if (tp +      fn) > 0 else 0.0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        metrics_report[f"Class {c}"] = {
            "IoU (Jaccard)": f"{100 * iou:.2f}%",
            "Precision": f"{100 * precision:.2f}%",
            "Recall": f"{100 * recall:.2f}%",
            "F1-Score": f"{100 * f1:.2f}%",
        }

    import pandas as pd
    return pd.DataFrame(metrics_report).T

# Load testing data

In [6]:
test_patches = np.load(data_dir + "/test_patches.npy")
test_patches = torch.from_numpy(test_patches)

x_test = test_patches[:, :-1]
y_test = test_patches[:, -1]

print(x_test.shape, y_test.shape)

torch.Size([676, 13, 56, 56]) torch.Size([676, 56, 56])


# Test Model

In [ ]:
results = evaluate_model(test_patches, best_model)
results

,IoU (Jaccard),Precision,Recall,F1-Score
Class 0,94.18%,97.35%,96.65%,97.00%
Class 1,49.89%,63.99%,69.36%,66.57%


# Test multiple thresholds on validation data. Use informal settlement IoU as reference for performance.

In [ ]:
val_patches = np.load(data_dir + "/val_patches.npy")
val_patches = torch.from_numpy(val_patches).float()

x_val = val_patches[:, :-1]
y_val = val_patches[:, -1]

print(x_val.shape, y_val.shape)

torch.Size([676, 13, 56, 56]) torch.Size([676, 56, 56])


In [ ]:
for thresh in range(1, 11, 1):
    threshold = thresh/10
    results = evaluate_model(val_patches, best_model, threshold)
    print("threshold:", threshold, "informal class IoU:", results["IoU (Jaccard)"]["Class 1"])

threshold: 0.1 informal class IoU: 54.55%
threshold: 0.2 informal class IoU: 54.54%
threshold: 0.3 informal class IoU: 54.53%
threshold: 0.4 informal class IoU: 54.50%
threshold: 0.5 informal class IoU: 54.46%
threshold: 0.6 informal class IoU: 54.41%
threshold: 0.7 informal class IoU: 54.34%
threshold: 0.8 informal class IoU: 54.24%
threshold: 0.9 informal class IoU: 54.01%
threshold: 1.0 informal class IoU: 44.73%


# Evaluate on test data with best threshold: 0.1

In [15]:
results = evaluate_model(test_patches, best_model, 0.1)
results

,IoU (Jaccard),Precision,Recall,F1-Score
Class 0,94.06%,97.47%,96.41%,96.94%
Class 1,49.94%,62.85%,70.85%,66.61%


### Analysis

The final test-set evaluation shows that the model provides a substantially stronger ability to identify non-informal-settlement areas than informal-settlement areas. At the default threshold of 0.5, the model achieved an IoU of **94.18% for Class 0**, compared with **49.89% for Class 1**. The Class 0 precision, recall, and F1-score were also high at **97.35%, 96.65%, and 97.00%**, respectively. This indicates that the model is highly reliable when identifying pixels belonging to the non-informal-settlement class.


---



Performance on the informal-settlement class was more moderate, with a **49.89% IoU, 63.99% precision, 69.36% recall, and 66.57% F1-score**. The higher recall than precision indicates that the model was more successful at finding informal-settlement pixels than avoiding false informal-settlement predictions. In other words, the model tends to identify a relatively large proportion of the actual informal-settlement areas, but some of its positive predictions correspond to areas that are not labelled as informal settlements.


---


The informal-settlement IoU remained remarkably stable between thresholds of 0.1 and 0.9, ranging only from **54.55% to 54.01%**. The highest validation IoU occurred at a threshold of 0.1, but the improvement over the default 0.5 threshold was only **0.09 percentage points**. This indicates that threshold adjustment provided very little practical benefit. The substantial drop to **44.73% at a threshold of 1.0** further shows that requiring extremely high model confidence to classify a pixel as informal settlement substantially reduces segmentation performance.


---


Based on the validation results, a threshold of **0.1** was selected for the final test evaluation. However, the test results demonstrate why the distinction between a numerical improvement and a meaningful improvement is important. Compared with the default threshold of 0.5, the 0.1 threshold increased Class 1 IoU only marginally, from **49.89% to 49.94%**, and F1-score from **66.57% to 66.61%**. The main change was an increase in recall from **69.36% to 70.85%**, accompanied by a decrease in precision from **63.99% to 62.85%**. This is the expected trade-off when lowering the classification threshold: the model identifies more potential informal-settlement pixels, increasing the number of true positives but also increasing false positives.


---


These results provide a suitable conclusion to the deep learning component of the project. The model demonstrates **strong segmentation of non-informal-settlement areas and moderate segmentation of informal-settlement areas**, with the latter achieving approximately **50% IoU and 67% F1-score** on the test set. Its relatively high informal-settlement recall shows that it can identify a substantial proportion of the target areas, although the lower precision demonstrates that distinguishing informal settlements from visually similar non-informal areas remains challenging.


---


**The final model establishes a useful baseline for satellite-image-based informal-settlement segmentation, while the remaining errors highlight the difficulty of reliably distinguishing informal settlements across diverse urban environments.**